# Fase 1: Hough global en Google Colab


In [ ]:
%pip install nvcc4jupyter
%load_ext nvcc4jupyter
!nvidia-smi
!nvcc --version


## Archivos del proyecto


In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, os
uploaded = files.upload()
assert len(uploaded) == 1, "Sube solamente proyecto-colab.zip"
root = Path('/content/proyecto-hough')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for name in archive.namelist():
        assert (root / name).resolve().is_relative_to(root.resolve()), "Ruta ZIP invalida"
    archive.extractall(root)
os.chdir(root)
print('Proyecto:', root)


## Ejecución con nvcc4jupyter


In [ ]:
from nvcc4jupyter import set_defaults
set_defaults(compiler_args="-std=c++17 -O2 --fmad=false -I/content/proyecto-hough")


In [ ]:
%%cuda
#include "common/pgm.cpp"
#define main hough_program_main
#include "src/houghBase.cu"
#undef main
int main() {
    char program[] = "hough";
    char image[] = "/content/proyecto-hough/data/runway.pgm";
    char* args[] = {program, image};
    return hough_program_main(2, args);
}


## Validación y evidencia


In [ ]:
import subprocess, tempfile
from pathlib import Path
subprocess.run(['make'], check=True)
log = subprocess.check_output(['nvidia-smi'], text=True)
log += subprocess.check_output(['nvcc', '--version'], text=True)
run = subprocess.run(['./build/hough', 'data/runway.pgm'], text=True, capture_output=True)
print(run.stdout, run.stderr)
log += run.stdout + run.stderr
assert run.returncode == 0, 'La validacion GPU fallo'
with tempfile.TemporaryDirectory() as directory:
    for name, data in [('parcial', bytes([255]) * 21), ('negra', bytes(21))]:
        path = Path(directory) / (name + '.pgm')
        path.write_bytes(b'P5\n7 3\n255\n' + data)
        test = subprocess.run(['./build/hough', str(path)], text=True, capture_output=True)
        log += name + '\n' + test.stdout + test.stderr
        assert test.returncode == 0, test.stderr
    for args in [[], ['archivo-inexistente.pgm']]:
        test = subprocess.run(['./build/hough', *args], text=True, capture_output=True)
        log += test.stdout + test.stderr
        assert test.returncode != 0, 'Debe rechazar argumentos o archivos invalidos'
Path('results').mkdir(parents=True, exist_ok=True)
Path('results/fase1-colab.txt').write_text(log)
print('Pruebas completadas. Evidencia: results/fase1-colab.txt')
files.download('results/fase1-colab.txt')
